<a href="https://colab.research.google.com/github/ssykes-eth/ETH_273-0003-00L/blob/weekend_4/CVAE%20CX/notebook_CelebA_unsolved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Conditional VAE on CelebA: Generating and Manipulating Faces
In-Class Exercise

In [ ]:
# @title Setup {display-mode: "form"}
%cd /content
!rm -rf "CVAE CX"
!mkdir -p "CVAE CX/pretrained_models"
%cd "CVAE CX"

BASE = "https://raw.githubusercontent.com/eth-bmai-fs26/coding-exercises/main/CVAE%20CX"

!curl -sO {BASE}/data.py
!curl -sO {BASE}/models.py
!curl -sO {BASE}/utils.py

# checkpoint pretrained
for b in ["0.10","0.60","0.80","1.00","1.20","1.50","5.00"]:
    !curl -sL {BASE}/pretrained_models/celeba_cvae_beta{b}.pt -o pretrained_models/celeba_cvae_beta{b}.pt

!pip install -q datasets


In [ ]:
from utils import *
from models import *
from data import *
import torch.nn.functional as F

setup()

---
## Part 1: The Dataset

CelebA contains over 200,000 celebrity face images, each labelled with **40 binary attributes** (Smiling, Eyeglasses, Male, Blond_Hair, …). These attributes are **multi-label** — a face can be Smiling AND Young AND Wearing_Lipstick simultaneously.

We use 30 k images (≈ 15 % of the full dataset) to fit in Colab RAM. Images are cropped to a 128×128 face-centred patch and resized to 64×64.

In [ ]:
X_train, attrs_train, X_test, attrs_test = load_celeba(max_samples=30000)

print(f"\nAttribute names ({len(CELEBA_ATTR_NAMES)}):")
for i, name in enumerate(CELEBA_ATTR_NAMES):
    print(f"  {i:2d}. {name}", end="\t" if (i+1) % 4 != 0 else "\n")

In [ ]:
# @title Sample faces {display-mode: "form"}
fig, axes = plt.subplots(2, 8, figsize=(20, 5))
fig.suptitle("Sample faces from CelebA", fontsize=14, fontweight='bold')
for i in range(16):
    ax = axes[i // 8, i % 8]
    ax.imshow(np.clip(X_train[i].transpose(1, 2, 0), 0, 1))
    active = [CELEBA_ATTR_NAMES[j] for j in range(40) if attrs_train[i, j] == 1.0]
    ax.set_title(', '.join(active[:3]), fontsize=7)
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# @title Attribute frequency distribution {display-mode: "form"}
freq = attrs_train.mean(axis=0)
sorted_idx = np.argsort(freq)[::-1]
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(40), freq[sorted_idx], color='steelblue', edgecolor='black')
ax.set_xticks(range(40))
ax.set_xticklabels([CELEBA_ATTR_NAMES[i] for i in sorted_idx], rotation=90, fontsize=8)
ax.set_ylabel('Fraction of images', fontsize=12)
ax.set_title('CelebA Attribute Frequencies', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print(f"Most common:  {CELEBA_ATTR_NAMES[sorted_idx[0]]} ({freq[sorted_idx[0]]:.1%})")
print(f"Least common: {CELEBA_ATTR_NAMES[sorted_idx[-1]]} ({freq[sorted_idx[-1]]:.1%})")

---
## Part 2: How a CVAE Works

### The Big Picture

| | Autoencoder | VAE | Conditional VAE |
|---|---|---|---|
| Bottleneck | Fixed vector | Fuzzy region | Fuzzy region |
| Can generate new images? | No | Yes (random) | Yes (**on demand**) |
| Analogy | Photocopier | Artist painting freely | Artist following a brief |

Think of it as a studio with two specialists:
- **The Critic (Encoder):** looks at a face and summarises its *style* — bone structure, skin tone, lighting — into a short description (the latent vector **z**).
- **The Artist (Decoder):** receives the style description **plus a brief** (the 40 attributes: Smiling, Young, Eyeglasses…) and paints a new face matching both.

The brief is the *condition*. This lets you tell the network **what** to generate while z controls **how** it looks.

---

### Why the Latent Space Must Be 'Fuzzy'

A regular autoencoder maps each image to a single exact point in latent space. The decoder memorises those exact locations — pick a random point and you land in empty space.

A VAE maps each image to a **small cloud** (a probability distribution) instead of a point. Training forces all clouds to overlap near the centre. The result: the latent space is dense and smooth, so any random point decodes to something plausible — this is what enables generation.

---
## Part 3: Architecture — How Data Flows

The five methods you implement in TODO 1 correspond directly to the five steps below:

**Step 1 — conditional_input**
The encoder needs to see both the image *and* the attributes at once. The attributes vector (40 numbers) is expanded to fill a full spatial map and glued onto the image as extra channels: `(3ch image) + (40ch label map)` → `43ch input`.

**Step 2 — encode**
The 43-channel input passes through 4 convolutional blocks (64×64 → 32×32 → 16×16 → 8×8 → 4×4), is flattened, then projected to two vectors: **μ** (mean) and **log σ²** (log-variance). These describe where in latent space this image lives.

**Step 3 — reparametrize**
We sample a latent code: `z = μ + σ · ε` where `ε` is random noise. Writing it this way (instead of sampling z directly) keeps the computation graph intact so gradients can flow back to the encoder.

**Step 4 — decode**
`z` (128 numbers) is concatenated with the attributes (40 numbers), projected to a `256×4×4` feature map, then upsampled through 5 transposed conv blocks back to a `3×64×64` image.

**Step 5 — forward**
Chains the above: encode → reparametrize → decode. Returns the reconstructed image plus μ and log σ² (needed for the loss).

```
Image (3×64×64) ─┐
                  ├─► conditional_input ─► encode ─► μ, log σ²
Attributes (40) ──┘                                      │
                                                  reparametrize
                                                      z (128)
                                                         │
Attributes (40) ──────────────────────────────► decode ──┘
                                                   │
                                          Output image (3×64×64)
```

---
### TODO 1 — Complete the CelebAConvCVAE class

Five methods to implement. Hints are in the comments directly above each method.

In [ ]:
class CelebAConvCVAE(nn.Module):
    """
    Encoder : (43, 64, 64) -> mu, log_var  (latent_dim each)
    Decoder : (latent_dim + 40) -> (3, 64, 64)
    """
    def __init__(self, latent_dim=128, label_dim=40):
        super().__init__()
        self.latent_dim = latent_dim
        self.label_dim  = label_dim

        # -- ENCODER ---------------------------------------------------
        # 4 conv blocks: 32->64->128->256, kernel 3x3, stride 2
        # Spatial: 64->32->16->8->4
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(3 + label_dim, 32,  3, stride=2, padding=1),
            nn.BatchNorm2d(32),  nn.LeakyReLU(0.2),
            nn.Conv2d(32,  64,  3, stride=2, padding=1),
            nn.BatchNorm2d(64),  nn.LeakyReLU(0.2),
            nn.Conv2d(64,  128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
        )
        # Single linear: outputs mu || log_var concatenated, split in encode()
        self.fc_latent = nn.Linear(256 * 4 * 4, latent_dim * 2)

        # -- DECODER ---------------------------------------------------
        # 5 blocks: 4x stride-2 upsamplings + 1x stride-1 refinement
        # Spatial: 4->8->16->32->64->64
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim + label_dim, 256 * 4 * 4),
            nn.LeakyReLU(0.2),
        )
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(256, 256, 3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(256, 128, 3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(128, 64,  3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(64),  nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64,  32,  3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(32),  nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32,   3,  3, stride=1, padding=1),
            nn.Sigmoid(),
        )

    # ------------------------------------------------------------------
    # TODO 1a — conditional_input
    # Role: Appends the attribute vector as extra spatial channels to the image.
    # Hint: 1. Add two trailing dimensions to label: label.unsqueeze(-1).unsqueeze(-1)
    #          gives shape (B, 40, 1, 1)
    #       2. Expand those dims to match the image: .expand(-1, -1, x.size(2), x.size(3))
    #          gives shape (B, 40, 64, 64)
    #       3. Concatenate with x along the channel dimension (dim=1)
    #          gives shape (B, 43, 64, 64)
    # ------------------------------------------------------------------
    def conditional_input(self, x, label):
        """Broadcast the attribute label spatially and concatenate with the image."""
        # === BEGIN YOUR CODE (remove `pass` and write here) ===
        pass
        # === END YOUR CODE ===

    # ------------------------------------------------------------------
    # TODO 1b — reparametrize
    # Role: Samples a latent vector z while keeping the computation differentiable.
    # Hint: 1. Compute std from log_var:  std = torch.exp(0.5 * log_var)
    #       2. Draw random noise:          eps = torch.randn_like(std)
    #       3. Return the sample:          mu + std * eps
    # ------------------------------------------------------------------
    def reparametrize(self, mu, log_var):
        """Sample z = mu + std * eps, eps ~ N(0,I), keeping the graph differentiable."""
        # === BEGIN YOUR CODE (remove `pass` and write here) ===
        pass
        # === END YOUR CODE ===

    # ------------------------------------------------------------------
    # TODO 1c — encode
    # Role: Maps an image and its attributes to the latent distribution parameters.
    # Hint: 1. Build the conditional input:  self.conditional_input(x, label)
    #       2. Pass through encoder_conv, then flatten to 1D
    #       3. Pass through fc_latent  ->  vector of size 2 * latent_dim
    #       4. Split in half: out[:, :self.latent_dim] is mu,
    #                         out[:, self.latent_dim:] is log_var
    # ------------------------------------------------------------------
    def encode(self, x, label):
        """Encode image + attributes to (mu, log_var) of the latent distribution."""
        # === BEGIN YOUR CODE (remove `pass` and write here) ===
        pass
        # === END YOUR CODE ===

    # ------------------------------------------------------------------
    # TODO 1d — decode
    # Role: Reconstructs an image from a latent code and a set of attributes.
    # Hint: 1. Concatenate z and label along dim=1
    #       2. Pass through decoder_fc
    #       3. Reshape to (batch_size, 256, 4, 4)  using .view(-1, 256, 4, 4)
    #       4. Pass through decoder_conv
    # ------------------------------------------------------------------
    def decode(self, z, label):
        """Decode latent z and attributes back to a (3, 64, 64) image."""
        # === BEGIN YOUR CODE (remove `pass` and write here) ===
        pass
        # === END YOUR CODE ===

    # ------------------------------------------------------------------
    # TODO 1e — forward
    # Role: Runs the full CVAE pipeline on a batch of images.
    # Hint: 1. mu, log_var = self.encode(x, label)
    #       2. z           = self.reparametrize(mu, log_var)
    #       3. recon_x     = self.decode(z, label)
    #       4. return recon_x, mu, log_var
    # ------------------------------------------------------------------
    def forward(self, x, label):
        """Full pipeline: encode image, sample z, decode back to pixels."""
        # === BEGIN YOUR CODE (remove `pass` and write here) ===
        pass
        # === END YOUR CODE ===

In [ ]:
LATENT_DIM = 128
cvae = CelebAConvCVAE(latent_dim=LATENT_DIM).to(DEVICE)
print(f"CelebAConvCVAE: {sum(p.numel() for p in cvae.parameters()):,} parameters")

---
### The Two-Part Loss

The model minimises one combined number at every training step:

```
Total loss  =  Reconstruction loss  +  β × KL loss
```

**Reconstruction loss (BCE)**
Measures pixel-by-pixel how different the output is from the input — lower means a more faithful copy. Technically it is binary cross-entropy summed over all pixels and averaged over the batch.

**KL loss**
Measures how far the encoder's clouds are from a standard centred Gaussian. It pulls all clouds toward the centre, keeping the latent space smooth and navigable. It has a clean closed-form expression in terms of the two numbers the encoder outputs for each image — the mean **μ** and the log-variance **log σ²**:

```python
kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / batch_size
```

Each term checks one dimension of latent space: `mu.pow(2)` penalises means that drift from zero; `log_var.exp()` penalises variances that deviate from 1; `1 + log_var` rewards non-zero variance — preventing the encoder from collapsing each cloud to a single point.

---

### The β Dial

Training balances two competing goals. β is the weight on the regularisation goal:

| β | What it means | Effect |
|---|---|---|
| β < 1  (e.g. 0.1) | Reconstruction dominates | Sharp, detailed images; latent space may have gaps — random sampling can fail |
| **β = 1** | Standard VAE — both goals equally weighted | The theoretical balance point; each unit of regularisation costs one unit of reconstruction quality |
| β > 1  (e.g. 5) | Regularisation dominates | Smooth, well-organised latent space; images become progressively blurrier as β grows |

---
### TODO 2 — Implement the CVAE Loss

Fill in `cvae_loss` below using the two terms above. The reconstruction loss is one line with `F.binary_cross_entropy(..., reduction='sum')`. The KL loss formula is given in the code block above — transcribe it directly.

In [ ]:
def cvae_loss(recon_x, x, mu, log_var, beta=1.0):
    batch_size = x.size(0)
    # === BEGIN YOUR CODE (remove `pass` and write here) ===
    # TODO 2a: reconstruction loss

    # TODO 2 KL divergence
    pass
    # === END YOUR CODE ===
    return recon_loss + beta * kl_loss, recon_loss, kl_loss

---
## Part 4: Training

Instead of starting from random weights, we **load a pretrained checkpoint** (100 epochs, β=1.50) and fine-tune for **5 more epochs** at β=1.50. This lets you observe real training dynamics without waiting for the full run.

The checkpoint lives in `pretrained_models/` inside the repo.

In [ ]:
# Load pretrained checkpoint (beta=1.50, 100 epochs)
ckpt_path = 'pretrained_models/celeba_cvae_beta1.50.pt'
ckpt = torch.load(ckpt_path, map_location=DEVICE)
cvae.load_state_dict(ckpt['model_state_dict'])
loss_history = ckpt.get('loss_history', [])
print(f"Loaded: beta={ckpt['beta']}, pretrained epochs={ckpt['epochs']}")

# Fine-tune for 5 more epochs at beta=1.50
# print("\nFine-tuning for 5 epochs (beta=1.50)...")
# new_history = train_cvae_celeba(
#     cvae, cvae_loss, X_train, attrs_train,
#     epochs=5, lr=1e-3, beta=1.50
# )
# loss_history = loss_history + new_history

In [ ]:
plot_losses(loss_history)

---
## Part 5: Reconstruction

**What reconstruction does:** passes a real image through the *full* forward pass (encode → reparametrize → decode). The encoder infers where in latent space the image lives; the decoder reconstructs it from there.

```python
# core of reconstruct_cvae_celeba()
recon, _, _ = model(X_t, a_t)   # encode -> reparametrize -> decode
```

The `_, _` discards `mu` and `log_var` — we only need the pixel output here. Reconstructions will be **slightly blurrier** than the originals: the KL term prevents the encoder from perfectly pinpointing each image in latent space, so there is always a small reconstruction gap.

In [ ]:
recon = reconstruct_cvae_celeba(cvae, X_test[:100], attrs_test[:100])
plot_original_vs_reconstructed_celeba(X_test[:100], recon,
                                      title="CelebA CVAE: Original vs Reconstructed")

In [ ]:
# @title Per-attribute reconstruction error {display-mode: "form"}
attr_mse = plot_per_attribute_mse(recon, X_test[:100], attrs_test[:100], CELEBA_ATTR_NAMES)

---
## Part 6: Conditional Generation

**Generation vs Reconstruction — a critical distinction:**

| | Reconstruction | Generation |
|---|---|---|
| z comes from | Encoder (inferred from a real image) | Sampled randomly from N(0, 0.75²I) |
| Encoder used? | Yes | **No** |
| Starting point | A real face | Nothing — pure noise |
| What varies | Fixed z, fixed attributes | Different z each time → different styles |

This is the CVAE's key superpower: we never need a real image to generate. The KL term is what makes this work — it ensures random z samples land in densely populated, well-decoded regions of latent space.

We sample with std=0.75 (not 1.0) to stay near the centre of the learned distribution, avoiding the tails where the model has seen fewer training examples.

```python
# core of generate_conditional_celeba()
z = torch.randn(n, latent_dim).to(DEVICE) * 0.75  # sample from prior
for name, val in attrs_dict.items():
    attrs[:, attr_names.index(name)] = float(val)  # set requested attributes
images = model.decode(z, attrs)                    # decoder only — encoder not involved
```

### TODO 3 — Generate faces with specific attributes

Use `generate_conditional_celeba(model, attrs_dict, attr_names, n, latent_dim, seed)`. Try at least 3 different attribute combinations.

The `seed` controls which z vectors are sampled — keep it the same across calls to compare attributes on the same faces, or change it to explore a different set of styles.

In [ ]:
generate_conditional_celeba(cvae, {"Smiling": 1, "Young": 1},
                            CELEBA_ATTR_NAMES, n=8, latent_dim=LATENT_DIM, seed=0)
generate_conditional_celeba(cvae, {"Male": 1, "Eyeglasses": 1},
                            CELEBA_ATTR_NAMES, n=8, latent_dim=LATENT_DIM, seed=0)
generate_conditional_celeba(cvae, {"Blond_Hair": 1, "Smiling": 1, "Young": 1},
                            CELEBA_ATTR_NAMES, n=8, latent_dim=LATENT_DIM, seed=0)
generate_conditional_celeba(cvae, {"Bald": 1, "Smiling": 1, "Male":1},
                            CELEBA_ATTR_NAMES, n=8, latent_dim=LATENT_DIM, seed=0)

---
## Part 7: Latent Space

Since attributes are provided as a separate condition, z should capture **residual style** — lighting, face shape, pose — not attribute identity. We project the 128-D latent space to 2D with PCA and colour by an attribute. A well-trained CVAE should show **significant overlap** between the two groups: the attribute is in the condition, not in z.

In [ ]:
visualise_latent_space_celeba(cvae, X_test, attrs_test, CELEBA_ATTR_NAMES,
                              color_by='Smiling', n_viz=3000)
visualise_latent_space_celeba(cvae, X_test, attrs_test, CELEBA_ATTR_NAMES,
                              color_by='Male', n_viz=3000)

---
## Part 8: Interpolation

We can smoothly morph between two faces by **walking a straight line** in latent space between their encoded means. Because the KL term keeps the space smooth, every point on that line decodes to a plausible face.

```python
# core of interpolate_latent_celeba()
mu1, _ = model.encode(x1, a1)   # encode image 1 -> latent mean (no reparametrisation)
mu2, _ = model.encode(x2, a2)   # encode image 2

for r in np.linspace(0, 1, n_steps):
    z   = (1 - r) * mu1 + r * mu2      # linear interpolation
    img = model.decode(z, attrs)        # decode at each step
```

Note we use **μ directly** (not a reparametrised sample) — this gives the most representative, stable point for each image.

**Fixed vs switched attributes:**
- Fixed: the identity stays the same; only style morphs
- Switched: at the midpoint the label flips to image 2's attributes; style continues morphing but identity jumps — demonstrating that z and the label are independent controls

---
### TODO 4 — Interpolate between two faces

Use `interpolate_latent_celeba(model, X_test, attrs_test, idx1, idx2, n_steps)`. Try both `switch_attrs=False` and `switch_attrs=True` and describe what you observe.

In [ ]:
# Fixed attributes
interpolate_latent_celeba(cvae, X_test, attrs_test, idx1=0, idx2=7, n_steps=10)

In [ ]:
# Attribute switch at midpoint
interpolate_latent_celeba(cvae, X_test, attrs_test, idx1=0, idx2=7,
                          n_steps=10, switch_attrs=True)

---
### Attribute Manipulation

The most powerful CelebA-specific feature: encode a real face, then decode with a **flipped attribute** while keeping z identical. The person's identity is roughly preserved; only the targeted attribute changes.

```python
# core of manipulate_attribute()
mu, _ = model.encode(x, a_orig)          # encode -> get style vector
recon_orig = model.decode(mu, a_orig)     # reconstruct normally

a_flip[:, attr_idx] = 1 - a_flip[:, attr_idx]  # flip the target attribute
recon_flip = model.decode(mu, a_flip)     # same z, different condition
```

This works because z and the label are **separate inputs** to the decoder — they are independently controllable by design.

In [ ]:
for attr in [
      "Mouth_Slightly_Open",
      "Smiling",
      "Eyeglasses",
      "Mustache",
      "No_Beard",
      "Pale_Skin",
      "Rosy_Cheeks",
      "Male",
      "Blond_Hair",
      "Bushy_Eyebrows",
      "Narrow_Eyes",
      "Heavy_Makeup",
  ]:
      manipulate_attribute(cvae, X_test, attrs_test, CELEBA_ATTR_NAMES,
                           idx=12, attr_to_change=attr)


---
## Part 9: The Effect of β — Comparing Pretrained Models

Seven models were pretrained for 100 epochs, each at a different β. We load them all and generate faces with the same attributes and the same z vectors so that differences come purely from β.

| β | Name | Expected behaviour |
|---|---|---|
| 0.10 | recon_dominant | Very sharp, may mishandle attributes |
| 0.60 | recon_favoured | Sharp, close to EleMisi's setting |
| 0.80 | slight_recon_bias | Good quality, slight reconstruction emphasis |
| 1.00 | balanced | Standard VAE balance point |
| 1.20 | slight_kl_bias | Slightly softer, more regular latent space |
| 1.50 | kl_favoured | Noticeably blurrier |
| 5.00 | kl_dominant | Very blurry, highly regularised |

The comparison uses **shared z vectors** across all β values so you're comparing the decoder's rendering quality, not random variation in z.

In [ ]:
beta_configs = [
    (0.1,  'celeba_cvae_beta0.10'),
    (0.6,  'celeba_cvae_beta0.60'),
    (0.8,  'celeba_cvae_beta0.80'),
    (1.0,  'celeba_cvae_beta1.00'),
    (1.2,  'celeba_cvae_beta1.20'),
    (1.5,  'celeba_cvae_beta1.50'),
    (5.0,  'celeba_cvae_beta5.00'),
]

beta_models = {}
for beta, name in beta_configs:
    ckpt = torch.load(f'pretrained_models/{name}.pt', map_location=DEVICE)
    m = CelebAConvCVAE(latent_dim=LATENT_DIM).to(DEVICE)
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval()
    beta_models[beta] = m
    print(f'Loaded beta={beta:.2f}: {name}')

In [ ]:
beta_sweep_grid_generate(beta_models, beta_configs, CELEBA_ATTR_NAMES,
                         latent_dim=LATENT_DIM,
                         target_attrs={"Smiling": 1, "Young": 1},
                         n_samples=6, seed=0)

In [ ]:
beta_sweep_grid_reconstruct(beta_models, beta_configs, X_test, attrs_test,
                            CELEBA_ATTR_NAMES, n_samples=6, seed=0)

---
## Part 10: Quantitative Evaluation

We train an attribute predictor on real CelebA images, then test it on CVAE-generated images. If the predictor correctly identifies the requested attributes, the CVAE is generating attribute-faithful faces. Common attributes (Smiling, Male, Young) should score higher than rare ones (Bald, Wearing_Hat).

In [ ]:
# @title Train predictor and evaluate {display-mode: "form"}
print("Training attribute predictor on real images...")
predictor = train_attribute_predictor(CelebAAttributePredictor, X_train, attrs_train, epochs=10)

print("\nEvaluating CVAE-generated images...")
per_attr_acc = evaluate_generated_celeba(cvae, predictor, CELEBA_ATTR_NAMES,
                                          latent_dim=LATENT_DIM, n_samples=1000)

---
## References

- **ConditionalVAE** — E. Misino, *Conditional Variational Autoencoder on CelebA*, GitHub, 2021. The architecture, training hyperparameters, and preprocessing used in this exercise are adapted from this implementation. [github.com/EleMisi/ConditionalVAE](https://github.com/EleMisi/ConditionalVAE)

- **β-VAE** — Higgins et al., *β-VAE: Learning Basic Visual Concepts with a Constrained Variational Framework*, ICLR 2017.

- **CelebA dataset** — Liu et al., *Deep Learning Face Attributes in the Wild*, ICCV 2015.